# 02 — Transform Subscriptions to Silver

## Purpose

Transform validated subscription events from Bronze into a clean, current-state Silver subscription table.

This notebook will:

- Read subscription events from the Bronze Delta table
- Standardize subscription attributes and data types
- Deduplicate source events
- Apply CDC using the latest event per subscription
- Exclude subscriptions whose latest operation is `DELETE`
- Validate customer ownership and subscription lifecycle rules
- Persist the final dataset through an idempotent Delta merge

## Sources

- `workspace.revenue_leakage_bronze.subscription_events`
- `workspace.revenue_leakage_silver.customers`

## Target

- `workspace.revenue_leakage_silver.subscriptions`

## 1. Load and Inspect Subscription Sources

Load the Bronze subscription events and the Silver customer reference table, then inspect the exact subscription schema, event distribution, and available customer population.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_SUBSCRIPTIONS_TABLE = (
    "workspace.revenue_leakage_bronze.subscription_events"
)

SILVER_CUSTOMERS_TABLE = (
    "workspace.revenue_leakage_silver.customers"
)

SILVER_SUBSCRIPTIONS_TABLE = (
    "workspace.revenue_leakage_silver.subscriptions"
)

bronze_subscription_events_df = spark.table(
    BRONZE_SUBSCRIPTIONS_TABLE
)

silver_customer_reference_df = spark.table(
    SILVER_CUSTOMERS_TABLE
)

print(
    f"Bronze subscription events: "
    f"{bronze_subscription_events_df.count():,}"
)

print(
    f"Silver customer references: "
    f"{silver_customer_reference_df.count():,}"
)

bronze_subscription_events_df.printSchema()

display(
    bronze_subscription_events_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

## 2. Standardize and Resolve Subscription CDC Events

Clean and standardize the Bronze subscription attributes, remove exact duplicate events, and resolve the latest state of each subscription.

Only subscriptions belonging to customers present in the current Silver customer table are retained, preserving referential integrity across the Silver layer.

In [0]:
subscription_cdc_window = (
    Window
    .partitionBy("subscription_id")
    .orderBy(
        F.col("event_timestamp").desc(),
        F.col(
            "_source_file_modification_time"
        ).desc(),
        F.col("_ingested_at").desc(),
        F.col("_record_hash").desc(),
    )
)

standardized_subscription_events_df = (
    bronze_subscription_events_df
    .dropDuplicates(["_record_hash"])
    .withColumn(
        "subscription_id",
        F.trim(F.col("subscription_id")),
    )
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id")),
    )
    .withColumn(
        "plan_id",
        F.trim(F.col("plan_id")),
    )
    .withColumn(
        "plan_name",
        F.trim(F.col("plan_name")),
    )
    .withColumn(
        "subscription_status",
        F.initcap(
            F.trim(F.col("subscription_status"))
        ),
    )
    .withColumn(
        "billing_frequency",
        F.initcap(
            F.trim(F.col("billing_frequency"))
        ),
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency"))),
    )
    .withColumn(
        "operation",
        F.upper(F.trim(F.col("operation"))),
    )
)

latest_subscription_events_df = (
    standardized_subscription_events_df
    .withColumn(
        "_cdc_rank",
        F.row_number().over(
            subscription_cdc_window
        ),
    )
    .filter(F.col("_cdc_rank") == 1)
    .drop("_cdc_rank")
)

active_subscription_states_df = (
    latest_subscription_events_df
    .filter(F.col("operation") != "DELETE")
)

silver_customer_keys_df = (
    silver_customer_reference_df
    .select("customer_id")
    .distinct()
)

orphan_subscription_states_df = (
    active_subscription_states_df
    .join(
        silver_customer_keys_df,
        on="customer_id",
        how="left_anti",
    )
)

silver_subscriptions_df = (
    active_subscription_states_df
    .join(
        silver_customer_keys_df,
        on="customer_id",
        how="inner",
    )
    .select(
        "subscription_id",
        "customer_id",
        "plan_id",
        "plan_name",
        "start_date",
        "end_date",
        "subscription_status",
        "billing_frequency",
        "billing_day",
        "base_monthly_price",
        "discount_percentage",
        "contracted_monthly_price",
        "contracted_billing_amount",
        "discount_start_date",
        "discount_end_date",
        "included_usage_units",
        "overage_unit_price",
        "payment_terms_days",
        "auto_renew",
        "currency",
        F.col("operation").alias(
            "last_operation"
        ),
        F.col("event_timestamp").alias(
            "last_event_timestamp"
        ),
        "snapshot_date",
        "_source_system",
        "_source_entity",
        "_source_file_path",
        "_record_hash",
        F.current_timestamp().alias(
            "_silver_processed_at"
        ),
    )
)

print(
    f"Bronze subscription events: "
    f"{bronze_subscription_events_df.count():,}"
)

print(
    f"Distinct Bronze events: "
    f"{standardized_subscription_events_df.count():,}"
)

print(
    f"Latest subscription states: "
    f"{latest_subscription_events_df.count():,}"
)

print(
    f"Subscriptions without a current customer: "
    f"{orphan_subscription_states_df.count():,}"
)

print(
    f"Eligible Silver subscriptions: "
    f"{silver_subscriptions_df.count():,}"
)

display(
    latest_subscription_events_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

## 3. Profile Subscription Business Domains

Inspect the standardized subscription status, plan, billing-frequency, and currency values before enforcing explicit Silver data-quality rules.

In [0]:
subscription_status_values = sorted(
    row["subscription_status"]
    for row in (
        silver_subscriptions_df
        .select("subscription_status")
        .distinct()
        .collect()
    )
)

plan_name_values = sorted(
    row["plan_name"]
    for row in (
        silver_subscriptions_df
        .select("plan_name")
        .distinct()
        .collect()
    )
)

billing_frequency_values = sorted(
    row["billing_frequency"]
    for row in (
        silver_subscriptions_df
        .select("billing_frequency")
        .distinct()
        .collect()
    )
)

currency_values = sorted(
    row["currency"]
    for row in (
        silver_subscriptions_df
        .select("currency")
        .distinct()
        .collect()
    )
)

print(
    f"Subscription statuses: "
    f"{subscription_status_values}"
)

print(
    f"Plan names: "
    f"{plan_name_values}"
)

print(
    f"Billing frequencies: "
    f"{billing_frequency_values}"
)

print(
    f"Currencies: "
    f"{currency_values}"
)

display(
    silver_subscriptions_df
    .groupBy(
        "subscription_status",
        "billing_frequency",
    )
    .count()
    .orderBy(
        "subscription_status",
        "billing_frequency",
    )
)

## 4. Validate the Current Silver Subscription State

Validate subscription uniqueness, customer ownership, business domains, lifecycle dates, billing configuration, and contracted pricing before persistence.

Subscriptions belonging to deleted customers are measured separately and intentionally excluded from the current-state Silver table.

In [0]:
EXPECTED_SILVER_SUBSCRIPTION_COUNT = 6_200
EXPECTED_EXCLUDED_SUBSCRIPTION_COUNT = 50

ALLOWED_SUBSCRIPTION_STATUSES = [
    "Active",
    "Cancelled",
    "Paused",
]

ALLOWED_PLAN_NAMES = [
    "Enterprise",
    "Growth",
    "Professional",
    "Starter",
]

ALLOWED_BILLING_FREQUENCIES = [
    "Annual",
    "Monthly",
]

ALLOWED_SUBSCRIPTION_CURRENCIES = [
    "USD",
]

required_subscription_columns = [
    "subscription_id",
    "customer_id",
    "plan_id",
    "plan_name",
    "start_date",
    "subscription_status",
    "billing_frequency",
    "billing_day",
    "base_monthly_price",
    "discount_percentage",
    "contracted_monthly_price",
    "contracted_billing_amount",
    "included_usage_units",
    "overage_unit_price",
    "payment_terms_days",
    "auto_renew",
    "currency",
    "last_event_timestamp",
    "snapshot_date",
]

required_subscription_field_is_missing = None

for column_name in required_subscription_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == ""
        )
    )

    required_subscription_field_is_missing = (
        missing_condition
        if required_subscription_field_is_missing
        is None
        else required_subscription_field_is_missing
        | missing_condition
    )

invalid_status_condition = (
    ~F.col("subscription_status").isin(
        ALLOWED_SUBSCRIPTION_STATUSES
    )
)

invalid_plan_condition = (
    ~F.col("plan_name").isin(
        ALLOWED_PLAN_NAMES
    )
)

invalid_billing_frequency_condition = (
    ~F.col("billing_frequency").isin(
        ALLOWED_BILLING_FREQUENCIES
    )
)

invalid_currency_condition = (
    ~F.col("currency").isin(
        ALLOWED_SUBSCRIPTION_CURRENCIES
    )
)

invalid_operation_condition = (
    ~F.col("last_operation").isin(
        "INSERT",
        "UPDATE",
    )
)

invalid_date_condition = (
    (
        F.col("start_date")
        > F.col("snapshot_date")
    )
    | (
        F.col("end_date").isNotNull()
        & (
            F.col("end_date")
            < F.col("start_date")
        )
    )
    | (
        F.col("discount_start_date").isNotNull()
        & F.col("discount_end_date").isNotNull()
        & (
            F.col("discount_end_date")
            < F.col("discount_start_date")
        )
    )
)

invalid_billing_configuration_condition = (
    ~F.col("billing_day").between(1, 28)
    | (F.col("included_usage_units") < 0)
    | (F.col("overage_unit_price") < 0)
    | (F.col("payment_terms_days") <= 0)
)

expected_monthly_price = F.round(
    F.col("base_monthly_price")
    * (
        F.lit(1)
        - (
            F.col("discount_percentage")
            / F.lit(100)
        )
    ),
    2,
)

expected_billing_amount = (
    F.when(
        F.col("billing_frequency") == "Annual",
        F.round(
            expected_monthly_price * F.lit(12),
            2,
        ),
    )
    .otherwise(expected_monthly_price)
)

invalid_pricing_condition = (
    (F.col("base_monthly_price") <= 0)
    | ~F.col("discount_percentage").between(
        0,
        100,
    )
    | (
        F.abs(
            F.col("contracted_monthly_price")
            - expected_monthly_price
        )
        > F.lit(0.01)
    )
    | (
        F.abs(
            F.col("contracted_billing_amount")
            - expected_billing_amount
        )
        > F.lit(0.01)
    )
)

subscription_validation_metrics = (
    silver_subscriptions_df
    .agg(
        F.count("*").alias(
            "silver_subscription_count"
        ),
        F.countDistinct(
            "subscription_id"
        ).alias(
            "distinct_subscription_count"
        ),
        F.sum(
            F.when(
                required_subscription_field_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_field_count"),
        F.sum(
            F.when(
                invalid_status_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_status_count"),
        F.sum(
            F.when(
                invalid_plan_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_plan_count"),
        F.sum(
            F.when(
                invalid_billing_frequency_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_billing_frequency_count"
        ),
        F.sum(
            F.when(
                invalid_currency_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_currency_count"),
        F.sum(
            F.when(
                invalid_operation_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_operation_count"),
        F.sum(
            F.when(
                invalid_date_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_date_count"),
        F.sum(
            F.when(
                invalid_billing_configuration_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_billing_configuration_count"
        ),
        F.sum(
            F.when(
                invalid_pricing_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_pricing_count"),
    )
    .first()
    .asDict()
)

duplicate_subscription_count = (
    subscription_validation_metrics[
        "silver_subscription_count"
    ]
    - subscription_validation_metrics[
        "distinct_subscription_count"
    ]
)

excluded_subscription_count = (
    orphan_subscription_states_df.count()
)

invalid_customer_reference_count = (
    silver_subscriptions_df
    .join(
        silver_customer_keys_df,
        on="customer_id",
        how="left_anti",
    )
    .count()
)

for metric_name, metric_value in (
    subscription_validation_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,}"
    )

print(
    f"duplicate_subscription_count: "
    f"{duplicate_subscription_count:,}"
)

print(
    f"excluded_deleted_customer_subscriptions: "
    f"{excluded_subscription_count:,}"
)

print(
    f"invalid_customer_reference_count: "
    f"{invalid_customer_reference_count:,}"
)

assert (
    subscription_validation_metrics[
        "silver_subscription_count"
    ]
    == EXPECTED_SILVER_SUBSCRIPTION_COUNT
), "Unexpected Silver subscription count."

assert duplicate_subscription_count == 0, (
    "Duplicate subscription IDs detected."
)

assert (
    excluded_subscription_count
    == EXPECTED_EXCLUDED_SUBSCRIPTION_COUNT
), "Unexpected excluded subscription count."

assert invalid_customer_reference_count == 0, (
    "Invalid customer references detected."
)

for metric_name in [
    "null_required_field_count",
    "invalid_status_count",
    "invalid_plan_count",
    "invalid_billing_frequency_count",
    "invalid_currency_count",
    "invalid_operation_count",
    "invalid_date_count",
    "invalid_billing_configuration_count",
    "invalid_pricing_count",
]:
    assert (
        subscription_validation_metrics[
            metric_name
        ]
        == 0
    ), f"Validation failed: {metric_name}"

print(
    "Silver subscription validation completed successfully."
)

## 5. Persist the Silver Subscription Table

Persist the validated current-state subscriptions as a managed Delta table.

The merge uses `subscription_id` as the business key and changes a record only when its source record hash has changed. New subscriptions are inserted, changed subscriptions are updated, and records absent from the resolved current state are deleted.

In [0]:
spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS
    workspace.revenue_leakage_silver
    """
)

silver_subscriptions_df.createOrReplaceTempView(
    "silver_subscription_updates"
)

if spark.catalog.tableExists(
    SILVER_SUBSCRIPTIONS_TABLE
):
    spark.sql(
        f"""
        MERGE INTO
          {SILVER_SUBSCRIPTIONS_TABLE} AS target
        USING
          silver_subscription_updates AS source
        ON
          target.subscription_id
          = source.subscription_id

        WHEN MATCHED
          AND target._record_hash
              <> source._record_hash
        THEN
          UPDATE SET *

        WHEN NOT MATCHED THEN
          INSERT *

        WHEN NOT MATCHED BY SOURCE THEN
          DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        silver_subscriptions_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            SILVER_SUBSCRIPTIONS_TABLE
        )
    )

    write_method = (
        "Initial Delta table creation"
    )

saved_silver_subscriptions_df = spark.table(
    SILVER_SUBSCRIPTIONS_TABLE
)

saved_subscription_count = (
    saved_silver_subscriptions_df.count()
)

saved_distinct_subscription_count = (
    saved_silver_subscriptions_df
    .select("subscription_id")
    .distinct()
    .count()
)

saved_invalid_customer_reference_count = (
    saved_silver_subscriptions_df
    .join(
        silver_customer_keys_df,
        on="customer_id",
        how="left_anti",
    )
    .count()
)

assert (
    saved_subscription_count
    == EXPECTED_SILVER_SUBSCRIPTION_COUNT
), "Saved Silver subscription count is incorrect."

assert (
    saved_distinct_subscription_count
    == saved_subscription_count
), "Saved table contains duplicate subscriptions."

assert (
    saved_invalid_customer_reference_count
    == 0
), "Saved table contains invalid customers."

print(
    f"Write method: {write_method}"
)

print(
    f"Silver table: "
    f"{SILVER_SUBSCRIPTIONS_TABLE}"
)

print(
    f"Saved Silver subscriptions: "
    f"{saved_subscription_count:,}"
)

print(
    f"Saved distinct subscription IDs: "
    f"{saved_distinct_subscription_count:,}"
)

print(
    f"Invalid saved customer references: "
    f"{saved_invalid_customer_reference_count:,}"
)

display(
    saved_silver_subscriptions_df
    .groupBy(
        "subscription_status",
        "plan_name",
    )
    .count()
    .orderBy(
        "subscription_status",
        "plan_name",
    )
)

## 6. Validate Idempotent Reprocessing

Reapply the same resolved subscription state through the Delta merge.

A successful rerun must produce zero inserts, updates, and deletes while preserving the same unique subscription population.

In [0]:
rows_before_subscription_rerun = (
    spark.table(
        SILVER_SUBSCRIPTIONS_TABLE
    )
    .count()
)

spark.sql(
    f"""
    MERGE INTO
      {SILVER_SUBSCRIPTIONS_TABLE} AS target
    USING
      silver_subscription_updates AS source
    ON
      target.subscription_id
      = source.subscription_id

    WHEN MATCHED
      AND target._record_hash
          <> source._record_hash
    THEN
      UPDATE SET *

    WHEN NOT MATCHED THEN
      INSERT *

    WHEN NOT MATCHED BY SOURCE THEN
      DELETE
    """
)

latest_subscription_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {SILVER_SUBSCRIPTIONS_TABLE}
        LIMIT 1
        """
    )
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

latest_subscription_history_row = (
    latest_subscription_history_df.first()
)

subscription_operation_metrics = (
    latest_subscription_history_row[
        "operationMetrics"
    ]
    or {}
)

rows_inserted = int(
    subscription_operation_metrics.get(
        "numTargetRowsInserted",
        "0",
    )
)

rows_updated = int(
    subscription_operation_metrics.get(
        "numTargetRowsUpdated",
        "0",
    )
)

rows_deleted = int(
    subscription_operation_metrics.get(
        "numTargetRowsDeleted",
        "0",
    )
)

subscriptions_after_rerun_df = spark.table(
    SILVER_SUBSCRIPTIONS_TABLE
)

rows_after_subscription_rerun = (
    subscriptions_after_rerun_df.count()
)

distinct_subscriptions_after_rerun = (
    subscriptions_after_rerun_df
    .select("subscription_id")
    .distinct()
    .count()
)

duplicate_subscriptions_after_rerun = (
    rows_after_subscription_rerun
    - distinct_subscriptions_after_rerun
)

assert (
    latest_subscription_history_row[
        "operation"
    ]
    == "MERGE"
), "Latest Delta operation was not MERGE."

assert rows_inserted == 0, (
    "Idempotency failed: rows were inserted."
)

assert rows_updated == 0, (
    "Idempotency failed: rows were updated."
)

assert rows_deleted == 0, (
    "Idempotency failed: rows were deleted."
)

assert (
    rows_before_subscription_rerun
    == EXPECTED_SILVER_SUBSCRIPTION_COUNT
), "Unexpected count before rerun."

assert (
    rows_after_subscription_rerun
    == EXPECTED_SILVER_SUBSCRIPTION_COUNT
), "Unexpected count after rerun."

assert (
    duplicate_subscriptions_after_rerun
    == 0
), "Duplicate subscriptions detected."

print(
    f"Rows before rerun: "
    f"{rows_before_subscription_rerun:,}"
)

print(
    f"Rows after rerun: "
    f"{rows_after_subscription_rerun:,}"
)

print(
    f"Rows inserted during rerun: "
    f"{rows_inserted:,}"
)

print(
    f"Rows updated during rerun: "
    f"{rows_updated:,}"
)

print(
    f"Rows deleted during rerun: "
    f"{rows_deleted:,}"
)

print(
    f"Duplicate subscriptions: "
    f"{duplicate_subscriptions_after_rerun:,}"
)

print(
    "Silver subscription transformation is idempotent."
)

display(
    latest_subscription_history_df
)

## Result

The subscription Silver transformation completed successfully:

- 6,550 Bronze subscription events processed
- 6,250 latest subscription states resolved
- 50 subscriptions belonging to deleted customers excluded
- 6,200 valid current-state subscriptions persisted
- Zero duplicate subscription identifiers
- Zero invalid customer references
- Zero lifecycle, domain, billing, or pricing violations
- Delta merge rerun produced zero changes
- Target confirmed as a managed Delta table